# 07 — Labels LLM (Gemini) a partir des rapports

Les rapports servent a **fabriquer** les 12 labels. Le submit Kaggle reste **vision only** (pas de Report au test).

Protocole fige :
1. Coller `GEMINI_API_KEY` dans `.env` (racine du repo).
2. Bake-off sur les **58 gold** : lexique vs `gemini-3.1-pro-preview`.
   (`gemini-2.5-pro` est ferme aux nouvelles cles.)
3. Si le LLM bat le lexique, labelliser les 4 349 avec **ce** modele.
4. Un **seul** modele labellise les 4 349. Pas de mix. Pas d'iteration du prompt sur l'or.
5. Filet lexique : desaccord 0/1 → `NaN` (masque), pas un second vote.
6. Pointer `configs/baseline.yaml` vers `data/labels/train_val_labels_llm.csv`, re-entrainer (recette v5, sans `pos_weight`).

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from src.nlp.gemini_labeler import SYSTEM_PROMPT, get_api_key, JSON_TO_TARGET

print('cles JSON :', list(JSON_TO_TARGET))
print('prompt :', len(SYSTEM_PROMPT), 'caracteres')
print('GEMINI_API_KEY :', 'ok' if get_api_key() else 'ABSENTE — remplir .env')

cles JSON : ['ACL', 'MCL', 'MedialMeniscus', 'LateralMeniscus', 'MedialOA', 'LateralOA', 'PFOA', 'Effusion', 'Synovitis', 'Bakers', 'Contusion', 'Fracture']
prompt : 3980 caracteres
GEMINI_API_KEY : ABSENTE — remplir .env


## Bake-off gold (58 etudes)

Depuis un terminal a la racine :

```
python -X utf8 -m src.nlp.gold_bakeoff --dry-run
python -X utf8 -m src.nlp.gold_bakeoff
```

Sorties : `data/labels/llm_gold_gemini-3.1-pro-preview.csv`, `llm_gold_bakeoff.json`.
Le cache (`data/labels/llm_cache/<modele>/`) permet de relancer sans re-payer.

In [2]:
import json
from pathlib import Path

p = Path.cwd().parent / 'data/labels/llm_gold_bakeoff.json'
if p.exists():
    d = json.loads(p.read_text(encoding='utf-8'))
    print('n_gold :', d['n_gold'])
    print('recommandation production :', d['recommendation'])
    for name, s in d['summaries'].items():
        print(f"{name:32s}  AUC={s['macro_AUC']:.3f}  F1={s['macro_F1']:.3f}  fid={s['macro_fidelite']:.3f}  cov={s['couverture_moy']:.2f}")
else:
    print('Pas encore de bake-off. Lancer : python -X utf8 -m src.nlp.gold_bakeoff')

Pas encore de bake-off. Lancer : python -X utf8 -m src.nlp.gold_bakeoff


## Corpus 4 349 (un modele)

Si le bake-off recommande 3.1 Preview, changer `production_model` dans `configs/llm_labels.yaml` **avant** ce run.

```
python -X utf8 -m src.nlp.label_corpus --workers 2
```

Interruption = OK (cache). Relancer la meme commande.

Puis dans `configs/baseline.yaml` :
`labels_csv: data/labels/train_val_labels_llm.csv`